In [7]:
!pip install 'gymnasium[atari]' --break-system-packages
!pip install 'gymnasium[accept-rom-license]' --break-system-packages

!pip install ale-py --break-system-packages

!pip install autorom[accept-rom-license] --break-system-packages
AutoROM --accept-license

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 1.7 MB/s eta 0:00:001.8 MB/s eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for AutoROM.accept-rom-license: filename=autorom_accept_rom_license-0.6.1-py3-none-any.whl size=446709 sha256=56ad0a37a79caa678871612010632e758004d861df9b89246eece58bd9931812
  Stored in directory: /home/soham/.cache/pip/wheels/99/f1/ff/c6966c034a8259164bdc9deb4d1ea839f119474638100e6645
Successfully built AutoROM.accept-rom-license


NameError: name 'AutoROM' is not defined

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from collections import deque
import random
import cv2
import matplotlib.pyplot as plt
from matplotlib import animation

In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class DQNNetwork(nn.Module):
    """Deep Q-Network architecture for Atari games"""
    def __init__(self, n_actions):
        super(DQNNetwork, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )
        
        self.fc = nn.Sequential(
            nn.Linear(7 * 7 * 64, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions)
        )
    
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

class ReplayBuffer:
    """Experience replay buffer for storing transitions"""
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards), 
                np.array(next_states), np.array(dones))
    
    def __len__(self):
        return len(self.buffer)

class FrameStack:
    """Stack frames for temporal information"""
    def __init__(self, n_frames=4):
        self.n_frames = n_frames
        self.frames = deque(maxlen=n_frames)
    
    def reset(self, frame):
        for _ in range(self.n_frames):
            self.frames.append(frame)
        return np.stack(self.frames, axis=0)
    
    def append(self, frame):
        self.frames.append(frame)
        return np.stack(self.frames, axis=0)

def preprocess_frame(frame):
    """Preprocess frame: grayscale, resize to 84x84"""
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
    return resized.astype(np.float32) / 255.0

class DQNAgent:
    """DQN Agent with epsilon-greedy policy"""
    def __init__(self, n_actions, lr=1e-4, gamma=0.99, epsilon_start=1.0, 
                 epsilon_end=0.1, epsilon_decay=1000000, buffer_size=100000):
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.steps = 0
        
        # Q-network and target network
        self.q_network = DQNNetwork(n_actions).to(device)
        self.target_network = DQNNetwork(n_actions).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.replay_buffer = ReplayBuffer(buffer_size)
        
    def select_action(self, state):
        """Epsilon-greedy action selection"""
        self.steps += 1
        self.epsilon = max(self.epsilon_end, 
                          self.epsilon_start - (self.steps / self.epsilon_decay))
        
        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)
        else:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                q_values = self.q_network(state_t)
                return q_values.argmax(1).item()
    
    def train_step(self, batch_size=32):
        """Perform one training step"""
        if len(self.replay_buffer) < batch_size:
            return None
        
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(batch_size)
        
        states = torch.FloatTensor(states).to(device)
        actions = torch.LongTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).to(device)
        
        # Current Q values
        current_q = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        
        # Target Q values
        with torch.no_grad():
            next_q = self.target_network(next_states).max(1)[0]
            target_q = rewards + (1 - dones) * self.gamma * next_q
        
        # Compute loss
        loss = nn.MSELoss()(current_q, target_q)
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), 10)
        self.optimizer.step()
        
        return loss.item()
    
    def update_target_network(self):
        """Update target network with Q-network weights"""
        self.target_network.load_state_dict(self.q_network.state_dict())

def train_dqn(env_name='ALE/Breakout-v5', n_episodes=1000, batch_size=32, 
              target_update=10000, save_interval=100, display_interval=10):
    """Train DQN agent on Atari game"""
    env = gym.make(env_name, render_mode='rgb_array')
    n_actions = env.action_space.n
    
    agent = DQNAgent(n_actions)
    frame_stack = FrameStack(n_frames=4)
    
    episode_rewards = []
    episode_losses = []
    best_reward = -float('inf')
    
    print(f"Training DQN on {env_name}")
    print(f"Number of actions: {n_actions}")
    print(f"Device: {device}\n")
    
    for episode in range(n_episodes):
        obs, _ = env.reset()
        frame = preprocess_frame(obs)
        state = frame_stack.reset(frame)
        
        episode_reward = 0
        episode_loss = []
        done = False
        truncated = False
        
        while not (done or truncated):
            # Select and perform action
            action = agent.select_action(state)
            next_obs, reward, done, truncated, _ = env.step(action)
            
            # Process next state
            next_frame = preprocess_frame(next_obs)
            next_state = frame_stack.append(next_frame)
            
            # Store transition
            agent.replay_buffer.push(state, action, reward, next_state, done or truncated)
            
            # Train agent
            loss = agent.train_step(batch_size)
            if loss is not None:
                episode_loss.append(loss)
            
            # Update target network
            if agent.steps % target_update == 0:
                agent.update_target_network()
                print(f"Target network updated at step {agent.steps}")
            
            state = next_state
            episode_reward += reward
        
        episode_rewards.append(episode_reward)
        avg_loss = np.mean(episode_loss) if episode_loss else 0
        episode_losses.append(avg_loss)
        
        # Logging
        if episode % display_interval == 0:
            avg_reward = np.mean(episode_rewards[-100:]) if len(episode_rewards) >= 100 else np.mean(episode_rewards)
            print(f"Episode {episode}/{n_episodes} | Reward: {episode_reward:.2f} | "
                  f"Avg(100): {avg_reward:.2f} | Loss: {avg_loss:.4f} | "
                  f"Epsilon: {agent.epsilon:.3f} | Steps: {agent.steps}")
        
        # Save best model
        if episode_reward > best_reward:
            best_reward = episode_reward
            torch.save(agent.q_network.state_dict(), f'dqn_{env_name.replace("/", "_")}_best.pth')
        
        # Periodic save
        if episode % save_interval == 0 and episode > 0:
            torch.save(agent.q_network.state_dict(), f'dqn_{env_name.replace("/", "_")}_ep{episode}.pth')
    
    env.close()
    return agent, episode_rewards, episode_losses

def visualize_agent(env_name='ALE/Breakout-v5', model_path=None, n_episodes=5):
    """Visualize trained agent playing the game"""
    env = gym.make(env_name, render_mode='rgb_array')
    n_actions = env.action_space.n
    
    agent = DQNAgent(n_actions)
    if model_path:
        agent.q_network.load_state_dict(torch.load(model_path, map_location=device))
    agent.epsilon = 0.05  # Small epsilon for visualization
    
    frame_stack = FrameStack(n_frames=4)
    
    for episode in range(n_episodes):
        obs, _ = env.reset()
        frame = preprocess_frame(obs)
        state = frame_stack.reset(frame)
        
        frames = []
        episode_reward = 0
        done = False
        truncated = False
        
        while not (done or truncated):
            # Render and save frame
            rgb_frame = env.render()
            frames.append(rgb_frame)
            
            # Select action
            action = agent.select_action(state)
            
            # Step environment
            next_obs, reward, done, truncated, _ = env.step(action)
            next_frame = preprocess_frame(next_obs)
            state = frame_stack.append(next_frame)
            episode_reward += reward
        
        print(f"Episode {episode + 1} | Reward: {episode_reward}")
        
        # Display frames
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.axis('off')
        img = ax.imshow(frames[0])
        
        def animate(i):
            img.set_data(frames[i])
            ax.set_title(f'Episode {episode + 1} | Frame {i}/{len(frames)} | Reward: {episode_reward:.0f}')
            return [img]
        
        anim = animation.FuncAnimation(fig, animate, frames=len(frames), 
                                      interval=50, blit=True, repeat=False)
        plt.show()
    
    env.close()

def plot_training_results(episode_rewards, episode_losses):
    """Plot training metrics"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot rewards
    ax1.plot(episode_rewards, alpha=0.3, label='Episode Reward')
    if len(episode_rewards) >= 100:
        moving_avg = [np.mean(episode_rewards[max(0, i-100):i+1]) 
                     for i in range(len(episode_rewards))]
        ax1.plot(moving_avg, label='Moving Average (100)')
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Reward')
    ax1.set_title('Training Rewards')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot losses
    ax2.plot(episode_losses, alpha=0.5)
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Loss')
    ax2.set_title('Training Loss')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

Using device: cpu


In [6]:
if __name__ == "__main__":
    # Example usage - train the agent
    print("Starting DQN training...")
    print("Note: This will take several hours to train properly.")
    print("For quick testing, reduce n_episodes to 50-100.\n")
    
    # Train (use fewer episodes for testing)
    agent, rewards, losses = train_dqn(
        env_name='ALE/Breakout-v5',
        n_episodes=100,  # Increase to 1000+ for real training
        batch_size=32,
        target_update=1000,
        display_interval=5
    )
    
    # Plot results
    plot_training_results(rewards, losses)
    
    # Visualize trained agent
    print("\nVisualizing trained agent...")
    visualize_agent(
        env_name='ALE/Breakout-v5',
        model_path='dqn_ALE_Breakout-v5_best.pth',
        n_episodes=3
    )

Starting DQN training...
Note: This will take several hours to train properly.
For quick testing, reduce n_episodes to 50-100.



NamespaceNotFound: Namespace ALE not found. Have you installed the proper package for ALE?